# NB05 v2 — Build03 Agentic Regulation Search · Retrieval Parity → Q7 Official Rerun

**Scientific AI Builder | Regulatory Agent | 2026-10-19**

## Research objective
Test whether **repeatable tool-driven retrieval** improves Q7 legal evidence coverage relative to frozen Build02 static Top-5 retrieval — **without changing the underlying retrieval algorithm or the frozen benchmark**.

**Workflow:**
1. Load frozen Build02 assets and NB04b context/configuration.
2. Recreate the **exact NB03 tokenizer**, embedding, BM25, RRF and diversification.
3. **STOP-GATE:** compare NB05 Top-5 vs frozen NB03 Top-5 for **all Q1–Q10 (50/50 ordered chunk IDs)**. Do not run official Q7 unless parity passes.
4. Register `search_regulation()` as an OpenAI Agents SDK `function_tool`.
5. Run Q7 using the **same answer-neutral shared benchmark context** and model/reasoning effort as NB04b.
6. Save final answer, retrieval logs, parity audit and comparison draft. **Preserve** the original Q7 pilot result.

### Frozen baselines and controlled factors
- Same 5 EU legislation files and `chunks_legal_aware.json`.
- Same `benchmark_questions_v1.json` and frozen Verified Reference/checklist.
- Same `gpt-6-luna`, `reasoning_effort=medium` as NB04b.
- Same *shared benchmark case context*; the Build03 tool-use instructions are a **declared treatment difference**.
- **Build02 retrieval remains untouched and frozen**; NB05 reimplements NB03 only to reuse the exact retriever as an agent-callable function.
- Q7 pilot (`data/build03_q7_smoke_test.json`) is **not overwritten**.

> **Important:** Better tool-call coverage ≠ proven answer correctness. Correct / Grounded / Complete are reviewed using the frozen checklist; this notebook does **not** auto-grade these with an LLM.

## 0. Dependencies, kernel, and API key

From the project repository root, install once if necessary. Restart the Jupyter kernel **after** any package changes. Keep your API key out of the notebook source and GitHub.

Install only when needed (avoid upgrading an already working Anaconda environment).

In [ ]:
# Resolve paths from the repository root, including nested Jupyter kernel CWDs.
import os as _repo_os
from pathlib import Path as _RepoPath
_repo_start = _RepoPath(_repo_os.environ.get("REGULATORY_AGENT_ROOT", _RepoPath.cwd())).expanduser().resolve()
REPO_ROOT = next((p for p in (_repo_start, *_repo_start.parents)
                  if (p / "data/benchmark/benchmark_questions_v1.json").is_file()
                  and (p / "notebooks/build04/NB09_v3.ipynb").is_file()), None)
if REPO_ROOT is None:
    raise FileNotFoundError("Set REGULATORY_AGENT_ROOT to the Regulatory_agent checkout.")
_repo_os.chdir(REPO_ROOT)

# Install only if required; then restart Kernel and run from the beginning.
# %pip install "openai-agents" "sentence-transformers" "rank-bm25" "numpy" "pandas"

In [2]:

import os, json, re, hashlib, platform
from pathlib import Path
from datetime import datetime, timezone
from getpass import getpass

import numpy as np
import pandas as pd
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from agents import Agent, Runner, ModelSettings, function_tool

# ============================================================
# 1. Project directory
# ============================================================

DATA_DIR = Path("data")

assert DATA_DIR.is_dir(), (
    "Run this notebook from the Regulatory_agent "
    "repository root (data/ missing)."
)

# ============================================================
# 2. OpenAI API Key setup
# ============================================================

if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass(
        "Enter your OpenAI API Key: "
    )

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY is required.")

# ============================================================
# 3. Model configuration
# ============================================================

MODEL_NAME = "gpt-6-luna"
REASONING_EFFORT = "medium"

print("Python:", platform.python_version())
print("Model:", MODEL_NAME)
print("Reasoning effort:", REASONING_EFFORT)
print("API key available:", bool(os.getenv("OPENAI_API_KEY")))

print("\nEnvironment setup completed.")


Python: 3.13.5
Model: gpt-6-luna
Reasoning effort: medium
API key available: True

Environment setup completed.


## 1. Load and fingerprint the frozen baseline

The reference outputs are **not regenerated or modified**:
- `data/chunks_legal_aware.json`
- `data/benchmark_questions_v1.json`
- `data/retrieval_config_nb03_final_hybrid_static_rag.json`
- `data/retrieval_results_nb03_final_hybrid_static_rag.csv`
- `data/nb04b_context_repaired_experiment_config.json`
- `data/nb04b_context_repaired_openai_luna_answers_comparison.csv`

These files must exist in your local clone (pull from GitHub if required).

In [3]:
FILES = {
    "chunks": __import__("pathlib").Path("data/processed/chunks_legal_aware.json"),
    "benchmark": __import__("pathlib").Path("data/benchmark/benchmark_questions_v1.json"),
    "retrieval_config": __import__("pathlib").Path("data/processed/retrieval_config_nb03_final_hybrid_static_rag.json"),
    "nb03_results": __import__("pathlib").Path("data/processed/retrieval_results_nb03_final_hybrid_static_rag.csv"),
    "nb04b_config": __import__("pathlib").Path("results/build01_02/nb04b_context_repaired_experiment_config.json"),
    "nb04b_answers": __import__("pathlib").Path("results/build01_02/nb04b_context_repaired_openai_luna_answers_comparison.csv"),
}
for name, path in FILES.items():
    if not path.is_file():
        raise FileNotFoundError(f"Missing {name}: {path}. Pull the complete data/ directory from GitHub.")

legal_chunks = json.loads(FILES["chunks"].read_text(encoding="utf-8"))
benchmark = json.loads(FILES["benchmark"].read_text(encoding="utf-8"))
retrieval_config = json.loads(FILES["retrieval_config"].read_text(encoding="utf-8"))
nb04b_config = json.loads(FILES["nb04b_config"].read_text(encoding="utf-8"))
nb03_df = pd.read_csv(FILES["nb03_results"], encoding="utf-8-sig")
comparison_df = pd.read_csv(FILES["nb04b_answers"])
questions = benchmark["questions"]

benchmark_hash = hashlib.sha256(json.dumps(
    questions, ensure_ascii=False, sort_keys=True
).encode("utf-8")).hexdigest()

assert list(questions) == [f"Q{i}" for i in range(1,11)]
assert benchmark_hash == retrieval_config["benchmark_sha256"]
assert retrieval_config["retrieval_method"] == "hybrid_dense_bm25_rrf_diverse_v1"
assert retrieval_config["rrf_k"] == 60
assert retrieval_config["candidate_depth_per_retriever"] == 50
assert retrieval_config["diversification"]["max_per_group"] == 2
assert retrieval_config["top_k"] == 5
assert len(legal_chunks) == 1383
assert len(nb03_df) == 50 and set(nb03_df["qid"]) == set(questions)
assert set(nb03_df["benchmark_sha256"].astype(str)) == {benchmark_hash}
assert nb04b_config["model"] == MODEL_NAME
assert nb04b_config["reasoning_effort"] == REASONING_EFFORT
assert nb04b_config["retrieval_frozen"] is True
assert comparison_df["qid"].is_unique

print("Benchmark:", benchmark["benchmark_id"])
print("SHA256:", benchmark_hash)
print("Legal chunks:", len(legal_chunks))
print("Frozen NB03 rows:", len(nb03_df))
print("NB04b model:", nb04b_config["model"])
print("NB04b reasoning:", nb04b_config["reasoning_effort"])

Benchmark: regulatory_agent_v1_notion_2026-10-08
SHA256: 2ee3bf32133c31bc62514cfa85e116bd5dd846a89b282a1a6f62f1ec849c3e4e
Legal chunks: 1383
Frozen NB03 rows: 50
NB04b model: gpt-6-luna
NB04b reasoning: medium


## 2. Rebuild NB03 retriever **exactly**

In the first NB05, BM25 used `re.findall(r"\b[\w./()-]+\b", ...)`, which **was not NB03's tokenizer**. This is fixed below.

The source for the tokenizer, scoring and selection is **`NB03_final_hybrid_static_RAG.ipynb`**, not a new tuning decision. The legal library, embedding model and source-grouping are unchanged.

**No benchmark-specific query expansion, gold snippets, or article-number injection.**

In [4]:
def safe(value):
    return "" if value is None else str(value)


def searchable_text(chunk):
    parts = [
        safe(chunk.get("regulation_title")),
        safe(chunk.get("file")),
        safe(chunk.get("location")),
        safe(chunk.get("article")),
        safe(chunk.get("heading")),
        safe(chunk.get("text")),
    ]
    return " | ".join(p for p in parts if p).strip()


# EXACT NB03 TOKENIZER
TOKEN_RE = re.compile(r"[A-Za-zÀ-ÿ0-9]+(?:[.,][0-9]+)?%?")
def tokenize(text):
    return TOKEN_RE.findall(text.casefold())


legal_texts = [searchable_text(c) for c in legal_chunks]
assert len(legal_texts) == len(legal_chunks)
assert all(legal_texts)

EMBEDDING_MODEL = retrieval_config["embedding_model"]
embedding_model = SentenceTransformer(EMBEDDING_MODEL)
legal_embeddings = embedding_model.encode(
    legal_texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True,
)
bm25 = BM25Okapi([tokenize(t) for t in legal_texts])
print("NB03-equivalent Dense + BM25 ready; model:", EMBEDDING_MODEL)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/44 [00:00<?, ?it/s]

NB03-equivalent Dense + BM25 ready; model: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2


In [5]:
RRF_K = int(retrieval_config["rrf_k"])
CANDIDATE_DEPTH = int(retrieval_config["candidate_depth_per_retriever"])
MAX_PER_SOURCE_GROUP = int(retrieval_config["diversification"]["max_per_group"])


def source_group(chunk):
    file_name = safe(chunk.get("file"))
    article = safe(chunk.get("article"))
    location = safe(chunk.get("location"))
    legal_unit = article if article else location
    return (file_name, legal_unit)


def hybrid_search(query: str, top_k: int = 5):
    """Single-search NB03-equivalent hybrid search, with configurable Top-K.

    The agent may repeat this *same* retriever with different queries.
    """
    query = str(query).strip()
    if not query:
        raise ValueError("query must be non-empty")
    if not 1 <= int(top_k) <= 10:
        raise ValueError("top_k must be in 1..10")

    q_emb = embedding_model.encode(
        [query], convert_to_numpy=True, normalize_embeddings=True,
        show_progress_bar=False
    )[0]
    dense_scores = legal_embeddings @ q_emb
    bm25_scores = np.asarray(bm25.get_scores(tokenize(query)), dtype=float)
    dense_order = np.argsort(dense_scores)[::-1][:CANDIDATE_DEPTH]
    bm25_order = np.argsort(bm25_scores)[::-1][:CANDIDATE_DEPTH]

    fused_scores = {}
    for rank, idx in enumerate(dense_order, start=1):
        idx = int(idx)
        fused_scores[idx] = fused_scores.get(idx, 0.0) + 1.0 / (RRF_K + rank)
    for rank, idx in enumerate(bm25_order, start=1):
        idx = int(idx)
        fused_scores[idx] = fused_scores.get(idx, 0.0) + 1.0 / (RRF_K + rank)
    fused_order = sorted(fused_scores, key=fused_scores.get, reverse=True)

    selected, counts = [], {}
    for idx in fused_order:
        chunk = legal_chunks[idx]
        group = source_group(chunk)
        if counts.get(group, 0) >= MAX_PER_SOURCE_GROUP:
            continue
        selected.append(int(idx))
        counts[group] = counts.get(group, 0) + 1
        if len(selected) == int(top_k):
            break

    return [
        {
            "rank": rank,
            "chunk_index": idx,
            "chunk_id": str(legal_chunks[idx].get("chunk_id", "")),
            "rrf_score": float(fused_scores[idx]),
            "file": legal_chunks[idx].get("file"),
            "regulation_title": legal_chunks[idx].get("regulation_title"),
            "article": legal_chunks[idx].get("article"),
            "location": legal_chunks[idx].get("location"),
            "heading": legal_chunks[idx].get("heading"),
            "text": legal_chunks[idx].get("text"),
        }
        for rank, idx in enumerate(selected, start=1)
    ]

print("hybrid_search ready")

hybrid_search ready


## 3. STOP-GATE: NB03 ↔ NB05 parity audit (Q1–Q10)

Exact **rank-ordered** chunk IDs must match for all 10 benchmark questions. Counts alone are not enough.

- **PASS 50/50:** continue to Part 4.
- **FAIL:** stop. Inspect tokenizer, library ordering, versions and ranking; do not treat subsequent Q7 as a controlled comparison.

The frozen NB03 CSV is never modified. A new diagnostic artifact is saved as `data/build03_v2_retrieval_parity_audit.csv`.

In [6]:
PARITY_CSV = __import__("pathlib").Path("experiments/build03_diagnostics/data/build03_v2_retrieval_parity_audit.csv")
audit_rows = []
parity_by_question = {}

for qid, question in questions.items():
    baseline = (nb03_df[nb03_df["qid"] == qid]
                .sort_values("rank"))
    expected = baseline["chunk_id"].astype(str).tolist()
    assert len(expected) == 5, (qid, "NB03 baseline must contain five rows")
    actual_rows = hybrid_search(question, top_k=5)
    actual = [r["chunk_id"] for r in actual_rows]
    parity_by_question[qid] = expected == actual

    for rank, (baseline_id, new_id) in enumerate(zip(expected, actual), start=1):
        audit_rows.append({
            "qid": qid,
            "rank": rank,
            "nb03_chunk_id": baseline_id,
            "nb05_v2_chunk_id": new_id,
            "match": baseline_id == new_id,
        })
    print(f"{qid}: {'PASS' if parity_by_question[qid] else 'MISMATCH'}")

parity_df = pd.DataFrame(audit_rows)
parity_df.to_csv(PARITY_CSV, index=False, encoding="utf-8-sig")
matched = int(parity_df["match"].sum())
PARITY_OK = bool(matched == 50 and all(parity_by_question.values()))
print(f"\nPARITY: {matched}/50 ordered chunks")
print("Saved:", PARITY_CSV)
if not PARITY_OK:
    display(parity_df[~parity_df["match"]])
    print("STOP: Do not run official Q7 until parity is fixed.")
else:
    print("PASS: NB05 v2 agrees with frozen NB03 for all Q1–Q10 Top-5 results.")

Q1: PASS
Q2: PASS
Q3: PASS
Q4: PASS
Q5: PASS
Q6: PASS
Q7: PASS
Q8: PASS
Q9: PASS
Q10: PASS

PARITY: 50/50 ordered chunks
Saved: data\build03_v2_retrieval_parity_audit.csv
PASS: NB05 v2 agrees with frozen NB03 for all Q1–Q10 Top-5 results.


In [ ]:
assert PARITY_OK, (
    "RETRIEVAL PARITY GATE FAILED. Do not run official Q7. "
    "Review build03_v2_retrieval_parity_audit.csv first."
)

## 4. Register dynamic `search_regulation()` function tool

Every call uses the **same NB03-equivalent hybrid search**. The agent may change `query` and call it repeatedly. For fair interpretation, `top_k` and the number of tool calls are logged: these are **Build03-specific retrieval-budget differences** compared with Build02 fixed Top-5.

We log the chunk IDs, legal location and retrieved text. This permits independent evidence audits later; the first NB05 pilot only logged locations.

In [7]:
TOOL_CALL_LOG = []

@function_tool
def search_regulation(query: str, top_k: int = 5) -> str:
    """Search the frozen five-law EU regulation library for legal evidence.

    Repeat with a narrower or complementary query if more legal evidence is needed.

    Args:
        query: Focused EU food-regulation search query.
        top_k: Number of ranked legal chunks to return (1–10, default 5).
    """
    top_k = max(1, min(int(top_k), 10))
    results = hybrid_search(query, top_k=top_k)
    TOOL_CALL_LOG.append({
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "query": query,
        "top_k": top_k,
        "results": results,
    })
    payload = [{
        "rank": r["rank"], "chunk_id": r["chunk_id"],
        "file": r["file"], "article": r["article"],
        "location": r["location"], "heading": r["heading"],
        "text": r["text"],
    } for r in results]
    return json.dumps(payload, ensure_ascii=False)

print("Registered OpenAI function_tool: search_regulation")

Registered OpenAI function_tool: search_regulation


## 5. Configure the **controlled Q7 official rerun**

**Same as NB04b**: `gpt-6-luna`, `medium` reasoning, same neutral case-context text, same frozen Q7 question and conservative answer requirements.

**Build03 intervention**: a single tool-enabled Agent may decide when and how often to retrieve evidence. Its tool-use rules are explicitly additional; Build02 received a fixed Top-5 evidence bundle.

The `benchmark_context` does **not** provide gold article numbers. The Q7 gold checklist is **evaluation-only** and must not enter the agent instructions.

NB04b also used a structured three-section answer. We request the same headings here to reduce output-format differences.

In [8]:
assert PARITY_OK

SHARED_CONTEXT = nb04b_config["benchmark_context"]
SHARED_SYSTEM_PROMPT = nb04b_config["system_prompt"]

# All new instructions are the *declared Build03 tool-use treatment*.
BUILD03_TOOL_RULES = """
You have access to search_regulation, which searches only the frozen five-law library.
Use it for substantive legal claims. If evidence is incomplete, you may search again
with a narrower or complementary query. Do not invent legal provisions, numbers,
mandatory actions, or requirements. Distinguish border-stage measures from
post-market operator duties if the evidence supports both.
Treat the local library as a historical snapshot; do not imply it proves current law.
Cite regulation number and precise article/annex when the retrieved text supports them.
Do not assume missing facts about the specific consignment.
""".strip()

AGENT_INSTRUCTIONS = "\n\n".join([
    SHARED_CONTEXT,
    SHARED_SYSTEM_PROMPT,
    BUILD03_TOOL_RULES,
])

regulatory_agent = Agent(
    name="Build03 Regulatory Research Agent v2",
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort=REASONING_EFFORT),
        max_tokens=int(nb04b_config["max_output_tokens"]),
    ),
    instructions=AGENT_INSTRUCTIONS,
    tools=[search_regulation],
)

Q7 = questions["Q7"]
print("Agent:", regulatory_agent.name)
print("Model:", regulatory_agent.model)
print("Reasoning effort:", REASONING_EFFORT)
print("Q7:", Q7)
print("Shared context starts:", SHARED_CONTEXT[:125].replace("\n", " "))

Agent: Build03 Regulatory Research Agent v2
Model: gpt-6-luna
Reasoning effort: medium
Q7: What regulatory actions may follow if aflatoxin levels exceed the applicable maximum level?
Shared context starts: All questions belong to one fixed EU food-regulation benchmark concerning the same case: a shipment of dried figs originating


## 6. Official Q7 rerun (one API-bearing call)

**Stop first if the parity cell did not print 50/50.** This is a new, controlled run rather than the 2026-10-19 pilot.

Jupyter uses an active event loop: run `await Runner.run(...)`, **not** `Runner.run_sync(...)`.

To protect API spending and the frozen experiment, do not rerun the cell casually. The result is saved immediately in Part 7.

In [9]:
assert PARITY_OK, "Parity gate failed"
if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter OPENAI_API_KEY: ")
assert os.getenv("OPENAI_API_KEY"), "API key missing"

TOOL_CALL_LOG.clear()

# This cell makes an actual paid API request.
q7_v2_result = await Runner.run(regulatory_agent, Q7, max_turns=15)
q7_v2_answer = str(q7_v2_result.final_output)

print("\n========== BUILD03 V2 Q7 OFFICIAL ANSWER ==========\n")
print(q7_v2_answer)
print("\nTool calls:", len(TOOL_CALL_LOG))
for i, call in enumerate(TOOL_CALL_LOG, start=1):
    locations = [r.get("article") or r.get("location") for r in call["results"]]
    print(f"{i}. top_k={call['top_k']} | {call['query']}")
    print("  Retrieved:", locations)


========== BUILD03 V2 Q7 OFFICIAL ANSWER ==========

## 1. Short answer

If the applicable aflatoxin maximum is exceeded, the food generally **must not be placed on the market or used as a food ingredient**. Depending on when and where the exceedance is established, possible consequences include official-control measures against the consignment and, if the food has already entered the market, restrictions, withdrawal or recall.

## 2. Legal basis and relevant conditions

- **Commission Regulation (EU) 2023/915, Article 2(1)** prohibits placing food listed in Annex I on the market, or using it as a raw material or ingredient, if it exceeds the applicable maximum level. Article 2(2) also prohibits mixing compliant food with food that exceeds the maximum.
- Under **Regulation (EU) 2017/625, Article 138**, where an official control establishes non-compliance, competent authorities may take appropriate measures. The listed possibilities include restricting or prohibiting placing goods on t

## 7. Save reproducible official result (never overwrite pilot)

Separate output file: `data/build03_q7_official_v2.json`.

The tool log includes full retrieved chunks; the file can be larger than the pilot log. Do not print/upload the API key. Store evidence provenance and run settings for later review.

In [10]:
assert PARITY_OK and "q7_v2_result" in globals(), "Run official Q7 before saving"

OFFICIAL_Q7_PATH = __import__("pathlib").Path("experiments/build03_diagnostics/data/build03_q7_official_v2.json")

run_record = {
    "build": "Build03",
    "experiment": "q7_official_v2_controlled_rerun",
    "status": "run_completed_evaluation_pending",
    "benchmark_id": benchmark["benchmark_id"],
    "benchmark_sha256": benchmark_hash,
    "question_id": "Q7",
    "question": Q7,
    "model": MODEL_NAME,
    "reasoning_effort": REASONING_EFFORT,
    "max_output_tokens": int(nb04b_config["max_output_tokens"]),
    "shared_benchmark_context": SHARED_CONTEXT,
    "shared_system_prompt": SHARED_SYSTEM_PROMPT,
    "build03_tool_rules": BUILD03_TOOL_RULES,
    "nb03_retrieval_config": retrieval_config,
    "retrieval_parity": {
        "passed": PARITY_OK,
        "matched_ordered_chunks": matched,
        "total_ordered_chunks": 50,
        "audit_file": str(PARITY_CSV),
    },
    "tool_call_count": len(TOOL_CALL_LOG),
    "tool_calls": TOOL_CALL_LOG,
    "final_output": q7_v2_answer,
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "pilot_preserved_at": "data/build03_q7_smoke_test.json",
}

# Fail safely rather than accidentally overwriting a prior official run.
if OFFICIAL_Q7_PATH.exists():
    raise FileExistsError(
        f"{OFFICIAL_Q7_PATH} exists already. Inspect it before explicitly choosing a new filename."
    )
OFFICIAL_Q7_PATH.write_text(
    json.dumps(run_record, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print("Saved:", OFFICIAL_Q7_PATH)

Saved: data\build03_q7_official_v2.json


## 8. Compare to **frozen NB04b Build02 Q7**

Use the stored Build02 RAG answer — **do not regenerate Build02**.

### Frozen Q7 checklist (Notion)
- [ ] Aflatoxin maximum-level exceedance recognized as non-compliance with 2023/915.
- [ ] Border-stage competent authority action under 2017/625 Art. 66 identified.
- [ ] Appropriate additional measures linked to Arts 67, 71, 72 under applicable conditions.
- [ ] Destruction **not automatic**; re-dispatch/treatment may be possible under applicable conditions.
- [ ] If already on market, 178/2002 Art. 19 duties: withdrawal, notification, recall where required.

### Evaluation labels
Use the existing frozen three-axis `Correct / Grounded / Complete` labels: `Yes`, `Partial`, `No`.

**Scoring guardrails**:
- Correct: distinguish legally correct statements from unsupported or overly broad assertions.
- Grounded: check *actual returned tool evidence text*, not merely the cited article number.
- Complete: apply **all** Q7 must-have checklist items, including border-stage measures.
- `Arts 66–72 coverage` is diagnostic and not a proxy for answer quality.
- Never use the gold checklist as generation prompt input.

The scoring template below is deliberately blank: a human/source audit must determine the final grades.

In [11]:
build02_q7 = comparison_df.loc[comparison_df["qid"] == "Q7", "rag_answer"].iloc[0]

print("========== FROZEN BUILD02 Q7 RAG ==========")
print(build02_q7)
print("\n========== BUILD03 Q7 OFFICIAL V2 ==========")
print(q7_v2_answer)

# These are the existing documented Build02 grades from the frozen Notion evaluation.
comparison_summary = {
    "question_id": "Q7",
    "build02_frozen": {
        "correct": "Partial", "grounded": "Yes", "complete": "No",
        "answer": build02_q7,
    },
    "build03_v2": {
        "correct": None, "grounded": None, "complete": None,
        "answer": q7_v2_answer,
        "review_status": "human_verified_reference_audit_required",
    },
    "comparison_conditions": {
        "same_question": True,
        "same_shared_benchmark_context": True,
        "same_model": True,
        "same_reasoning_effort": True,
        "same_retrieval_algorithm_on_frozen_benchmark": PARITY_OK,
        "different_instructions_and_retrieval_budget": True,
        "build02": "one fixed Top-5 retrieval",
        "build03": "agent chooses search queries, number of calls and Top-K (1..10)",
    },
}
print("\nAwaiting manual Correct / Grounded / Complete assessment.")

========== FROZEN BUILD02 Q7 RAG ==========
## 1. Short answer

The supplied evidence does not establish what regulatory actions may follow, or which actions would apply to this shipment. It gives aflatoxin limits for several categories of nuts and oilseeds, but not for dried figs, and it does not describe enforcement consequences [E1]–[E5].

## 2. Legal basis and relevant conditions

The cited extracts are from Regulation (EU) 2023/915 and concern maximum levels for other product categories. They do not support a conclusion about the applicable dried-fig limit or the actions available when that limit is exceeded.

## 3. Uncertainties / verification needed

The applicable maximum level for dried figs and the relevant enforcement and import-control rules must be verified in current law. Possible measures—such as refusing entry, requiring withdrawal or recall, or directing another disposition—cannot be confirmed from the supplied evidence.

========== BUILD03 Q7 OFFICIAL V2 ==========
##

## 9. Q7 diagnostic: where did the tool actually retrieve 2017/625 articles?

**Diagnostic only.** Test actual returned chunk **article metadata**, not keyword matches in model-generated search queries.

`66, 67, 71, 72` are held out of the agent instructions, used only for retrospective measurement.

In [12]:
Q7_TARGETS = ("66", "67", "71", "72")

def q7_target_coverage(log):
    retrieved = set()
    for call in log:
        for r in call["results"]:
            if "2017_625" not in str(r.get("file") or ""):
                continue
            art = str(r.get("article") or "").strip()
            m = re.fullmatch(r"Article\s+(\d+)", art, flags=re.IGNORECASE)
            if m:
                retrieved.add(m.group(1))
    return {f"Article {a}": a in retrieved for a in Q7_TARGETS}

coverage = q7_target_coverage(TOOL_CALL_LOG)
print("Retrieved 2017/625 target articles:", coverage)
print("Target coverage:", sum(coverage.values()), "/", len(coverage))

seen = []
for call in TOOL_CALL_LOG:
    for r in call["results"]:
        if "2017_625" in str(r.get("file") or ""):
            item = (r.get("article"), r.get("heading"))
            if item not in seen:
                seen.append(item)
print("All distinct 2017/625 retrieved article/heading pairs:")
for item in seen:
    print(" -", item)

Retrieved 2017/625 target articles: {'Article 66': False, 'Article 67': False, 'Article 71': False, 'Article 72': False}
Target coverage: 0 / 4
All distinct 2017/625 retrieved article/heading pairs:
 - ('Article 138', 'Actions in the event of established non-compliance')
 - ('Article 62', 'Withdrawal of the designation of border control posts')
 - ('Article 61', 'Withdrawal of approvals for, and re-designation of, existing border control entities')
 - ('Article 53', 'Official controls not performed at border control posts')
 - ('Article 48', 'Animals and goods exempted from official controls at border control posts')
 - ('Article 51', 'Specific rules for official controls at border control posts')
 - ('Article 63', 'Suspension of the designation of border control posts')
 - ('Article 49', 'Official controls at border control posts')
 - ('Article 64', 'Minimum requirements for border control posts')
 - ('Article 47', 'Animals and goods subject to official controls at border control post

## 10. Save an evaluation worksheet, not an invented final grade

This is the *next step*: review the corresponding regulation text for each assertion, enter the final Build03 labels, then save the completed evaluation table.

**If Q7 is still incomplete**, do not modify the frozen benchmark or silently inject target articles into prompts. A separate **border-consignment diagnostic** can be added later, clearly labeled out-of-benchmark.

The full Build03 Q1–Q10 experiment and Q8 current-law verification come afterward.

In [13]:
EVAL_CSV = __import__("pathlib").Path("experiments/build03_diagnostics/data/build03_v2_q7_evaluation_worksheet.csv")

worksheet = pd.DataFrame([
    {"qid":"Q7", "build":"Build02_frozen", "correct":"Partial", "grounded":"Yes", "complete":"No", "note":"Frozen NB04b / Notion grades"},
    {"qid":"Q7", "build":"Build03_v2", "correct":"", "grounded":"", "complete":"", "note":"TO REVIEW against frozen checklist and retrieved text"},
])
worksheet.to_csv(EVAL_CSV, index=False, encoding="utf-8-sig")
print("Saved worksheet:", EVAL_CSV)
display(worksheet)

Saved worksheet: data\build03_v2_q7_evaluation_worksheet.csv


,qid,build,correct,grounded,complete,note
0,Q7,Build02_frozen,Partial,Yes,No,Frozen NB04b / Notion grades
1,Q7,Build03_v2,,,,TO REVIEW against frozen checklist and retriev...


## Run completion checklist
- [ ] Exact original NB03 BM25 tokenization restored.
- [ ] **Parity 50/50** saved in `data/build03_v2_retrieval_parity_audit.csv`.
- [ ] Same benchmark context/model/reasoning effort applied to Build03 Q7.
- [ ] Official Q7 final answer and complete retrieved evidence saved separately.
- [ ] Frozen Build02 answer shown alongside Build03 answer.
- [ ] Correct / Grounded / Complete worksheet created **without automatic optimistic grading**.
- [ ] Pilot Q7 result remains untouched.

**Scientific interpretation:** Parity + answer evidence allows a defensible comparison, but a dynamic agent changes search budget (number and size of searches), and stochastic output may vary between runs. Report those differences rather than attributing all gains solely to “better reasoning.”